In [1]:
import pandas as pd
from IPython.display import display
import os

df = pd.read_csv('MegaDepth1500_summary.csv')

In [2]:
cols_to_exclude = ['Matcher', 'Min Time', 'Max Time', 'Avg Time']
score_cols = [c for c in df.columns if c not in cols_to_exclude]

df['Avg Score'] = df[score_cols].mean(axis=1)

df_sorted = df.sort_values('Avg Time')

base_cols = ['Matcher', 'Avg Time', 'Avg Score']
final_cols = base_cols + score_cols

In [3]:
def highlight_best_3(s, ascending=False):
    sorted_vals = sorted(s.unique(), reverse=not ascending)
    
    val_1 = sorted_vals[0] if len(sorted_vals) > 0 else None
    val_2 = sorted_vals[1] if len(sorted_vals) > 1 else None
    val_3 = sorted_vals[2] if len(sorted_vals) > 2 else None
    
    styles = []
    for val in s:
        if val == val_1:
            styles.append('background-color: #abebc6; color: black; font-weight: bold')
        elif val == val_2:
            styles.append('background-color: #aed6f1; color: black')
        elif val == val_3:
            styles.append('background-color: #f9e79f; color: black')
        else:
            styles.append('')
    return styles

In [4]:
print("Table: MegaDepth1500 Metrics (Sorted by Avg Time)")

styled_table = df_sorted[final_cols].style \
    .format(precision=2) \
    .hide(axis='index') \
    .apply(highlight_best_3, subset=['Avg Time'], ascending=True) \
    .apply(highlight_best_3, subset=['Avg Score'] + score_cols, ascending=False)

display(styled_table)

Table: MegaDepth1500 Metrics (Sorted by Avg Time)


Matcher,Avg Time,Avg Score,AUC@5,AUC@10,AUC@20
clidd-a48,6.37,52.68,39.75,53.58,64.71
clidd-n64,6.75,60.65,47.95,61.79,72.22
clidd-s64,7.15,61.84,49.05,62.88,73.59
clidd-t64,7.27,60.15,47.70,61.24,71.50
clidd-m64,7.68,64.42,51.77,65.46,76.02
clidd-l64,8.17,66.05,52.90,67.23,78.01
clidd-g128,11.46,67.72,54.77,68.96,79.43
clidd-e128,12.86,68.93,56.33,70.13,80.32
xfeat,14.55,49.79,35.89,50.45,63.04
clidd-u128,16.33,71.00,58.02,72.32,82.65


In [5]:
top3_thresholds = {}
cols_to_process = [c for c in final_cols if c != 'Matcher']

for col in cols_to_process:
    is_ascending = (col == 'Avg Time') 
    unique_vals = sorted(df_sorted[col].unique(), reverse=not is_ascending)
    top3_thresholds[col] = unique_vals[:3] # Simpan 3 nilai terbaik

colors = {0: '#abebc6', 1: '#aed6f1', 2: '#f9e79f'}

html_out = ['<table border="1" style="border-collapse: collapse; width: 100%; font-family: Arial, sans-serif;">']

html_out.append('<thead><tr style="background-color: #f2f2f2;">')
for col in final_cols:
    html_out.append(f'<th style="padding: 10px; text-align: center; border: 1px solid #ddd;">{col}</th>')
html_out.append('</tr></thead><tbody>')

for _, row in df_sorted[final_cols].iterrows():
    html_out.append('<tr>')
    for col in final_cols:
        val = row[col]
        display_val = f"{val:.2f}" if isinstance(val, float) else str(val)
        
        style = 'padding: 8px; text-align: center; border: 1px solid #ddd;'
        
        if col in top3_thresholds and val in top3_thresholds[col]:
            rank = top3_thresholds[col].index(val)
            if rank < 3:
                style += f' background-color: {colors[rank]}; color: black;'
                if rank == 0: style += ' font-weight: bold;'
        
        html_out.append(f'<td style="{style}">{display_val}</td>')
    html_out.append('</tr>')
html_out.append('</tbody></table>')

html_out.append('<br>')
html_out.append('<div style="font-family: Arial, sans-serif;">')
html_out.append('<strong>Color Information (Legend):</strong>')
html_out.append('<ul style="list-style: none; padding-left: 0;">')
html_out.append(f'<li style="margin-bottom: 5px;"><span style="background-color: {colors[0]}; border: 1px solid #ccc; padding: 2px 15px; margin-right: 8px;"></span> <strong>1st Best</strong> (Green)</li>')
html_out.append(f'<li style="margin-bottom: 5px;"><span style="background-color: {colors[1]}; border: 1px solid #ccc; padding: 2px 15px; margin-right: 8px;"></span> 2nd Best (Blue)</li>')
html_out.append(f'<li style="margin-bottom: 5px;"><span style="background-color: {colors[2]}; border: 1px solid #ccc; padding: 2px 15px; margin-right: 8px;"></span> 3rd Best (Yellow)</li>')
html_out.append('</ul></div>')

output_path = os.path.join('..', 'docs', 'MEGADEPTH_RESULT.md')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w') as f:
    f.write("# MegaDepth 1500 Evaluation Result\n\n")
    f.write("".join(html_out))

print(f"{os.path.abspath(output_path)} generated successfully.")

/home/ardyseto/Documents/GitHub/deep-matching-wrapper/docs/MEGADEPTH_RESULT.md generated successfully.
